In [1]:
import requests
import pandas as pd
import numpy as np

In [2]:
def get_all_players(years, teams):
    rolling_df = pd.DataFrame()
    for t in teams:
        for y in years:
            try:
                response = requests.get(f'https://api-web.nhle.com/v1/roster/{t}/{y}')
                if response.status_code == 200:
                  roster = response.json()
                else:
                    print(f'error: response code {response.status_code}')
                forwards = pd.DataFrame(roster['forwards'])
                defense = pd.DataFrame(roster['defensemen'])
                goalies = pd.DataFrame(roster['goalies'])
                all_pos_roster = pd.concat([forwards, defense, goalies])
                all_pos_roster["firstName"] = all_pos_roster["firstName"].apply(lambda x: next(iter(x.values())))
                all_pos_roster["lastName"] = all_pos_roster["lastName"].apply(lambda x: next(iter(x.values())))
                all_pos_roster['season'] = y
                all_pos_roster['team'] = t
                all_pos_roster = all_pos_roster.reset_index()
                all_pos_roster = all_pos_roster.drop(columns = ['index'])
                rolling_df = pd.concat([rolling_df, all_pos_roster])
            except:
                pass
    rolling_df = rolling_df.drop(columns = ['headshot', 'birthCity', 'birthCountry', 'birthStateProvince', 'heightInInches', 'weightInPounds'])
    rolling_df = rolling_df.reindex(columns = ['id', 'season', 'birthDate', 'team', 'firstName', 'lastName','sweaterNumber', 'positionCode', 'shootsCatches','heightInCentimeters', 'weightInKilograms'])
    return rolling_df

In [3]:
all_data = get_all_players(['20262027'], ["ANA", "BOS", "BUF", "CGY", "CAR", "CHI", "COL", "CBJ",
    "DAL", "DET", "EDM", "FLA", "LAK", "MIN", "MTL", "NSH",
    "NJD", "NYI", "NYR", "OTT", "PHI", "PIT", "SJS", "SEA",
    "STL", "TBL", "TOR", "UTA", "VAN", "VGK", "WPG", "WSH"])

In [4]:
all_data.drop_duplicates(subset=['id'], keep='first')
all_data.info()
bios = all_data[['id', 'firstName', 'lastName']]


<class 'pandas.DataFrame'>
Index: 796 entries, 0 to 24
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   id                   796 non-null    int64
 1   season               796 non-null    str  
 2   birthDate            796 non-null    str  
 3   team                 796 non-null    str  
 4   firstName            796 non-null    str  
 5   lastName             796 non-null    str  
 6   sweaterNumber        796 non-null    int64
 7   positionCode         796 non-null    str  
 8   shootsCatches        796 non-null    str  
 9   heightInCentimeters  796 non-null    int64
 10  weightInKilograms    796 non-null    int64
dtypes: int64(4), str(7)
memory usage: 74.6 KB


In [5]:
goalies = all_data[all_data['positionCode'] == 'G']
goalies = goalies.drop(columns = ['firstName', 'lastName'])
skaters = all_data[all_data['positionCode'] != 'G']
skaters = skaters.drop(columns = ['firstName', 'lastName'])

In [6]:
goalies.info()

<class 'pandas.DataFrame'>
Index: 74 entries, 23 to 24
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   id                   74 non-null     int64
 1   season               74 non-null     str  
 2   birthDate            74 non-null     str  
 3   team                 74 non-null     str  
 4   sweaterNumber        74 non-null     int64
 5   positionCode         74 non-null     str  
 6   shootsCatches        74 non-null     str  
 7   heightInCentimeters  74 non-null     int64
 8   weightInKilograms    74 non-null     int64
dtypes: int64(4), str(5)
memory usage: 5.8 KB


In [7]:
skaters.info()

<class 'pandas.DataFrame'>
Index: 722 entries, 0 to 22
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   id                   722 non-null    int64
 1   season               722 non-null    str  
 2   birthDate            722 non-null    str  
 3   team                 722 non-null    str  
 4   sweaterNumber        722 non-null    int64
 5   positionCode         722 non-null    str  
 6   shootsCatches        722 non-null    str  
 7   heightInCentimeters  722 non-null    int64
 8   weightInKilograms    722 non-null    int64
dtypes: int64(4), str(5)
memory usage: 56.4 KB


In [8]:
skaters.to_csv('skaters.csv', index = False)
goalies.to_csv('goalies.csv', index = False)

In [9]:
def get_player_stats_skater(PlayerId):
    response = requests.get(f'https://api-web.nhle.com/v1/player/{PlayerId}/landing')
    if response.status_code == 200:
        data = response.json()
    else:
        print(f'error: response code {response.status_code}')
    rolling_df = pd.DataFrame()
    for stats in data.get('seasonTotals'):
        if (stats.get('leagueAbbrev') == ('NHL')) and stats.get('gameTypeId') == 2:
            df = pd.DataFrame(stats)
            rolling_df = pd.concat([rolling_df, df])
    rolling_df = rolling_df.loc['default']
    rolling_df = rolling_df.reset_index(drop=True)
    rolling_df = rolling_df.drop(columns = ['gameTypeId', 'faceoffWinningPctg', 'gameWinningGoals', 'leagueAbbrev', 'otGoals', 'sequence', 'teamCommonName', 'teamName', 'teamPlaceNameWithPreposition'])
    rolling_df['season'] = rolling_df['season'].astype('str')
    rolling_df['skaterId'] = f'{PlayerId}'
    rolling_df = rolling_df.reindex(columns = ['skaterId', 'season', 'assists', 'avgToi', 'gamesPlayed', 'goals', 'pim', 'plusMinus', 'points', 'powerPlayGoals', 'powerPlayPoints', 'shootingPctg','shorthandedGoals', 'shorthandedPoints', 'shots'])
    return rolling_df

In [10]:
def get_player_stats_goalie(PlayerId):
    response = requests.get(f'https://api-web.nhle.com/v1/player/{PlayerId}/landing')
    if response.status_code == 200:
        data = response.json()
    else:
        print(f'error: response code {response.status_code}')
    rolling_df = pd.DataFrame()
    for stats in data.get('seasonTotals'):
        if (stats.get('leagueAbbrev') == ('NHL')) and stats.get('gameTypeId') == 2:
            df = pd.DataFrame(stats)
            rolling_df = pd.concat([rolling_df, df])
    rolling_df = rolling_df.drop_duplicates(subset=['season'], keep='first')
    rolling_df = rolling_df.reset_index(drop=True)
    rolling_df['skaterId'] = f'{PlayerId}'
    return rolling_df

In [11]:
def get_all_goalie_stats(goalieIds):
    summary_df = pd.DataFrame()
    for id in goalieIds:
        try:
            data = get_player_stats_goalie(id)
            summary_df = pd.concat([summary_df, data])
        except:
            pass
    summary_df = summary_df.drop(columns = ['assists', 'gameTypeId', 'goals', 'leagueAbbrev', 'sequence', 'teamCommonName', 'teamName', 'teamPlaceNameWithPreposition'])
    summary_df['season'] = summary_df['season'].astype(str)
    summary_df['saves'] = summary_df['savePctg']*summary_df['shotsAgainst']
    summary_df['saves'] = summary_df['saves'].round()
    summary_df = summary_df.reindex(columns = ['skaterId', 'season', 'gamesPlayed', 'gamesStarted', 'goalsAgainst', 'goalsAgainstAvg', 'losses', 'otLosses', 'saves', 'savePctg', 'shotsAgainst','shutouts', 'timeOnIce', 'wins'])
    try:
        summary_df['otLosses'] = summary_df['otLosses'].fillna(0)
    except:
        pass
    try:
        summary_df['savePctg'] = summary_df['savePctg'].fillna(1)
    except:
        pass
    try:
        summary_df['saves'] = summary_df['saves'].fillna(0)
    except:
        pass
    summary_df[['season']] = summary_df[['season']].astype(float)
    summary_df[['wins', 'losses']] = summary_df[['wins', 'losses']].fillna(0)
    summary_df[['season', 'gamesPlayed', 'gamesStarted', 'goalsAgainst', 'losses', 'otLosses', 'saves', 'shotsAgainst', 'shutouts', 'wins']] = summary_df[['season', 'gamesPlayed', 'gamesStarted', 'goalsAgainst', 'losses', 'otLosses', 'saves', 'shotsAgainst', 'shutouts', 'wins']].astype(int)
    summary_df[['season']] = summary_df[['season']].astype(str)
    return summary_df

In [12]:
goalie_summary = get_all_goalie_stats(goalies['id'].unique())
goalie_summary

,skaterId,season,gamesPlayed,gamesStarted,goalsAgainst,goalsAgainstAvg,losses,otLosses,saves,savePctg,shotsAgainst,shutouts,timeOnIce,wins
0,8476316,20142015,1,1,2,2.013422,1,0,49,0.960784,51,0,59:36,0
1,8476316,20152016,5,5,18,3.605807,4,1,124,0.873239,142,0,299:31,0
2,8476316,20162017,8,4,11,1.985260,1,0,142,0.928105,153,0,332:27,4
3,8476316,20172018,14,10,40,3.242218,7,1,303,0.883382,343,0,740:14,3
4,8476316,20182019,21,19,49,2.522988,6,2,603,0.924847,652,1,1165:17,13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2,8480313,20222023,37,36,96,2.652707,13,3,1036,0.915194,1132,2,2171:22,21
3,8480313,20232024,46,42,119,2.698956,14,5,1179,0.908320,1298,1,2645:28,25
4,8480313,20242025,43,42,105,2.485058,6,6,1067,0.910410,1172,2,2535:09,31
5,8480313,20252026,58,58,140,2.438281,21,6,1447,0.911783,1587,4,3445:03,31


In [34]:
def get_all_skater_stats(skaterIds):
    summary_df = pd.DataFrame()
    for id in skaterIds:
        try:
            data = get_player_stats_skater(id)
            summary_df = pd.concat([summary_df, data])
        except:
            pass
    summary_df[['season']] = summary_df[['season']].astype(str)
    return summary_df

In [35]:
skater_summary = get_all_skater_stats(skaters['id'].unique())

In [36]:
skater_summary.info()

<class 'pandas.DataFrame'>
Index: 6074 entries, 0 to 4
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   skaterId           6074 non-null   str    
 1   season             6074 non-null   str    
 2   assists            6074 non-null   int64  
 3   avgToi             6074 non-null   str    
 4   gamesPlayed        6074 non-null   int64  
 5   goals              6074 non-null   int64  
 6   pim                6074 non-null   int64  
 7   plusMinus          6074 non-null   int64  
 8   points             6074 non-null   int64  
 9   powerPlayGoals     6074 non-null   int64  
 10  powerPlayPoints    6074 non-null   int64  
 11  shootingPctg       6074 non-null   float64
 12  shorthandedGoals   6074 non-null   int64  
 13  shorthandedPoints  6074 non-null   int64  
 14  shots              6074 non-null   int64  
dtypes: float64(1), int64(11), str(3)
memory usage: 759.2 KB


In [37]:
goalie_summary.info()

<class 'pandas.DataFrame'>
Index: 569 entries, 0 to 6
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   skaterId         569 non-null    str    
 1   season           569 non-null    str    
 2   gamesPlayed      569 non-null    int64  
 3   gamesStarted     569 non-null    int64  
 4   goalsAgainst     569 non-null    int64  
 5   goalsAgainstAvg  569 non-null    float64
 6   losses           569 non-null    int64  
 7   otLosses         569 non-null    int64  
 8   saves            569 non-null    int64  
 9   savePctg         569 non-null    float64
 10  shotsAgainst     569 non-null    int64  
 11  shutouts         569 non-null    int64  
 12  timeOnIce        569 non-null    str    
 13  wins             569 non-null    int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 66.7 KB


In [38]:
skater_summary.to_csv('skater_summary.csv', index=False)
goalie_summary.to_csv('goalie_summary.csv', index=False)

In [13]:
bios.to_csv('nhl_bios.csv', index = False)